# Nizam Shaikh - 24198170

# Insurance Policy Chatbot Lab

This notebook builds a simple chatbot for one insurance policy PDF.

Students will learn how to:

- load a PDF
- split the document into chunks
- turn chunks into embeddings
- find the most relevant chunks
- ask an LLM to answer from the document

## Choose one provider

At the top of the notebook, choose:

- `openai`
- `gemini`
- `local`

The rest of the code stays the same.

## Step 1: Install packages

In [14]:
!pip install -q openai google-generativeai pypdf numpy ipywidgets sentence-transformers transformers accelerate

## Step 2: Choose provider and API key

Choose one provider:

- `PROVIDER = "openai"`
- `PROVIDER = "gemini"`
- `PROVIDER = "local"`

Then add only the key you need.

In [16]:
import os

# choose one: "local" (100% free, no API key), "gemini" or "openai"
PROVIDER = "local"

# ---- API keys (only needed for "openai" or "gemini") ----
# In Colab: click the key icon (Secrets) on the left, add the key,
# and switch "Notebook access" on. Never paste a real key into the notebook.
def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.getenv(name)

OPENAI_API_KEY = get_key("OPENAI_API_KEY") if PROVIDER == "openai" else None
GEMINI_API_KEY = get_key("GEMINI_API_KEY") if PROVIDER == "gemini" else None

# models
OPENAI_CHAT_MODEL = "gpt-4.1-mini"
OPENAI_EMBED_MODEL = "text-embedding-3-small"

GEMINI_CHAT_MODEL = "gemini-2.5-flash"
GEMINI_EMBED_MODEL = "models/gemini-embedding-001"

# free models that run inside Colab (no key, no billing)
LOCAL_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LOCAL_CHAT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# path to PDF (Colab stores uploaded files in /content)
DOCUMENT_PATH = "/content/Hospital Plans Rules_01Mar26_VHIMR2.pdf"

### Upload the PDF (Colab)

Run this cell and choose `Hospital Plans Rules_01Mar26_VHIMR2.pdf`.
It is skipped automatically if the file is already in Colab.

In [17]:
import os

if not os.path.exists(DOCUMENT_PATH):
    from google.colab import files
    uploaded = files.upload()          # pick the PDF in the dialog
    name = list(uploaded.keys())[0]
    DOCUMENT_PATH = "/content/" + name

print("Using:", DOCUMENT_PATH, "| exists:", os.path.exists(DOCUMENT_PATH))

Using: /content/Hospital Plans Rules_01Mar26_VHIMR2.pdf | exists: True


## Step 3: Import libraries

In [18]:
import numpy as np
from pypdf import PdfReader
from openai import OpenAI
import google.generativeai as genai

## Step 4: Connect to the model provider

In [19]:
if PROVIDER == "openai":
    assert OPENAI_API_KEY, "OPENAI_API_KEY missing - add it in Colab Secrets (key icon) and enable notebook access."
    client = OpenAI(api_key=OPENAI_API_KEY)

if PROVIDER == "gemini":
    assert GEMINI_API_KEY, "GEMINI_API_KEY missing - add it in Colab Secrets (key icon) and enable notebook access."
    genai.configure(api_key=GEMINI_API_KEY)

if PROVIDER == "local":
    # free: models are downloaded once and run inside Colab (first run takes a few minutes)
    import torch
    from sentence_transformers import SentenceTransformer
    from transformers import AutoTokenizer, AutoModelForCausalLM

    embed_model = SentenceTransformer(LOCAL_EMBED_MODEL)
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_CHAT_MODEL)
    chat_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_CHAT_MODEL, torch_dtype="auto", device_map="auto"
    )

print("Connected to", PROVIDER)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Connected to local


## Step 5: Load the PDF

We read the PDF page by page and combine all text into one string.

In [20]:
def load_document(path):
    text = ""

    if path.lower().endswith(".pdf"):
        reader = PdfReader(path)

        for page in reader.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"

    return text


doc_text = load_document(DOCUMENT_PATH)
print(doc_text[:1500])

Terms and Conditions
Applicable to new registrations or renewals on/or after 
1st October 2025, except for the following plans: 
PMI 18 11,  PMI 19 11 PMI 32 12, and, PMI 38 14 
which is applicable on/or after 1st November 2025.
Private Medical Insurance
Terms and Conditions
This document is applicable to new registrations 
or renewals on/or after 1st March 2026.  
See overleaf for exceptions
Exceptions to 1st March Effective Date
This document applies to new registrations or renewals for the plans listed below, from the dates shown.
APPLICABLE DATE PLAN NAME
2nd March  2026 PMI 01 10 PMI 05 11 PMI 06 10 PMI 10 11 PMI 14 10
PMI 17 10 PMI 18 11 PMI 21 11 PMI 24 10 PMI 29 12
PMI 39 14 PMI 50 10 PMI 56 10 PMI 59 10 PMI 60 10
PMI 63 10 PMI 64 10 PMI 65 10 PMI 66 10 PMI 68 10
PMI 70 10 EnhancedCare 
150
EnhancedCare 
150 Day to Day
EnhancedCare 
250
Enhanced Care 
250 Day to Day
EnhancedCare 
350
EnhancedCare 
350 Day to Day
3rd March 2026 PMI 63 10
1st April 2026 PMI 04 10
2nd April 2026 P

## Step 6: Chunk the document

Chunking means breaking one large document into smaller pieces.

This helps us search the document more easily.

### Simple idea

Big document → split into small pieces → find relevant pieces → send to LLM

---

### Example 1: No chunking (bad)

**Full document (very large):**

```
Policy covers hospitalization, surgery, maternity, dental...
Exclusions include cosmetic surgery, pre-existing diseases for 2 years...
Claim process requires documents A, B, C...
```

**Problem:**

- Too long  
- Hard for model to find the right part  
- Can miss important details  

---

### Example 2: With chunking (good)

**Chunk 1:**
```
Policy covers hospitalization and surgery expenses...
```

**Chunk 2:**
```
Exclusions include cosmetic surgery and pre-existing diseases...
```

**Chunk 3:**
```
Claim process requires submitting hospital bills and ID proof...
```

**Now:**

- Each chunk is small and focused  
- Easy to search  
- LLM gets only relevant parts  

In [21]:
def chunk_text(text, chunk_size=800, overlap=100):
    # clean extra spaces
    text = " ".join(text.split())

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)

        # move forward with overlap
        start = start + (chunk_size - overlap)

    return chunks


chunks = chunk_text(doc_text)

print("Number of chunks:", len(chunks))
print(chunks[0][:500])

Number of chunks: 180
Terms and Conditions Applicable to new registrations or renewals on/or after 1st October 2025, except for the following plans: PMI 18 11, PMI 19 11 PMI 32 12, and, PMI 38 14 which is applicable on/or after 1st November 2025. Private Medical Insurance Terms and Conditions This document is applicable to new registrations or renewals on/or after 1st March 2026. See overleaf for exceptions Exceptions to 1st March Effective Date This document applies to new registrations or renewals for the plans lis


## Step 7: Create embeddings

Embeddings convert text into numbers.

We do this for:
- document chunks
- the user question

Then we compare them.

We convert text into numbers so the computer can compare meaning.

---

### Step 1: Text input

```
Text 1: "Hospitalization is covered"
Text 2: "Surgery expenses are included"
Text 3: "Car insurance policy"
```

---

### Step 2: Convert to embeddings (numbers)

```
Text 1 → [0.12, 0.45, 0.33, ...]
Text 2 → [0.10, 0.47, 0.30, ...]
Text 3 → [0.80, 0.05, 0.10, ...]
```

---

### Step 3: Compare similarity

```
Text 1 vs Text 2 → Similar (both about health insurance)

Text 1 vs Text 3 → Not similar (health vs car insurance)
```

---

### Key idea

- Similar meaning → similar numbers  
- Different meaning → different numbers  

---

### In our chatbot

1. Convert document chunks into embeddings  
2. Convert user question into embedding  
3. Find the most similar chunks  
4. Send those chunks to the LLM  

## 6. Embeddings

We embed every chunk once, then use cosine similarity for retrieval.

In [22]:
def embed_texts(texts, batch_size=100):
    if PROVIDER == "openai":
        embeddings = []

        # send in batches so large PDFs do not hit request limits
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            response = client.embeddings.create(
                model=OPENAI_EMBED_MODEL,
                input=batch
            )
            for item in response.data:
                embeddings.append(item.embedding)

        return embeddings

    if PROVIDER == "gemini":
        embeddings = []

        for text in texts:
            response = genai.embed_content(
                model=GEMINI_EMBED_MODEL,
                content=text,
                task_type="retrieval_document"
            )
            embeddings.append(response["embedding"])

        return embeddings

    if PROVIDER == "local":
        vectors = embed_model.encode(texts, show_progress_bar=False)
        return [v.tolist() for v in vectors]

In [23]:
chunk_embeddings = embed_texts(chunks)
chunk_embeddings = np.array(chunk_embeddings)

print(chunk_embeddings.shape)

(180, 384)


## Step 8: Retrieve the top chunks

We:
1. convert the question into an embedding
2. compare it with each chunk
3. keep the highest scores

### Example: Finding the most relevant chunk

---

### Step 1: User question

```
"What does the policy cover?"
```

---

### Step 2: Chunks from document

```
Chunk 1: "Policy covers hospitalization and surgery expenses"

Chunk 2: "Exclusions include cosmetic surgery"

Chunk 3: "Car insurance covers vehicle damage"
```

---

### Step 3: Compare similarity (cosine similarity)

```
Question vs Chunk 1 → 0.92  (very similar)

Question vs Chunk 2 → 0.40  (less relevant)

Question vs Chunk 3 → 0.10  (not relevant)
```

---

### Step 4: Pick top results

```
Top result → Chunk 1
```

---

### Step 5: Send to LLM

```
Context:
"Policy covers hospitalization and surgery expenses"

Question:
"What does the policy cover?"
```

---

### Final Answer

```
The policy covers hospitalization and surgery expenses.
```

---

### Key idea

- Convert text into numbers (embeddings)
- Compare using cosine similarity
- Pick highest score
- Send best chunks to LLM

In [24]:
def cosine_similarity(a, b):
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    return dot_product / (norm_a * norm_b)


def retrieve_top_k(query, chunks, chunk_embeddings, k=4):
    if PROVIDER == "openai":
        query_embedding = embed_texts([query])[0]

    if PROVIDER == "gemini":
        response = genai.embed_content(
            model=GEMINI_EMBED_MODEL,
            content=query,
            task_type="retrieval_query"
        )
        query_embedding = response["embedding"]

    if PROVIDER == "local":
        query_embedding = embed_texts([query])[0]

    # score all chunks at once (same maths as cosine_similarity above, but faster)
    query_embedding = np.array(query_embedding)
    scores = (chunk_embeddings @ query_embedding) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )

    k = min(k, len(chunks))
    top_indices = np.argsort(scores)[::-1][:k]

    top_chunks = []

    for idx in top_indices:
        top_chunks.append({
            "chunk_id": int(idx),
            "score": round(float(scores[idx]), 4),
            "text": chunks[idx]
        })

    return top_chunks

## Step 9: Build the context

We combine the most relevant chunks into one context block.

In [25]:
def build_context(retrieved_chunks):
    context = ""

    for item in retrieved_chunks:
        context += "Chunk " + str(item["chunk_id"]) + ":\n"
        context += item["text"] + "\n\n"

    return context

## Step 10: Ask the model for the answer

The model must answer only from the document context.

In [26]:
SYSTEM_PROMPT = """
You are an insurance policy assistant.
Answer only from the provided policy context.
If the answer is not in the context, say:
I could not find that in the policy document.
""".strip()

In [27]:
def ask_llm(question, context):
    prompt = f"""
Context:
{context}

Question:
{question}

Answer using only the context above.
"""

    if PROVIDER == "openai":
        response = client.responses.create(
            model=OPENAI_CHAT_MODEL,
            instructions=SYSTEM_PROMPT,
            input=prompt
        )
        return response.output_text

    if PROVIDER == "gemini":
        model = genai.GenerativeModel(GEMINI_CHAT_MODEL)
        response = model.generate_content(SYSTEM_PROMPT + "\n\n" + prompt)
        return response.text

    if PROVIDER == "local":
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(chat_model.device)
        output_ids = chat_model.generate(
            **inputs, max_new_tokens=300, do_sample=False
        )
        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## Step 11: Final chatbot function

This function runs the full pipeline.

In [28]:
def answer_question(question, k=4):
    retrieved_chunks = retrieve_top_k(question, chunks, chunk_embeddings, k)
    context = build_context(retrieved_chunks)
    answer = ask_llm(question, context)

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks
    }

## Step 12: Test the chatbot

*   List item
*   List item

In [29]:
result = answer_question("Which hospitals are covered?")

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nTop chunks:")
for item in result["retrieved_chunks"]:
    print("\n-------------------")
    print("Chunk ID:", item["chunk_id"])
    print("Score:", item["score"])
    print(item["text"][:300])

Question:
Which hospitals are covered?

Answer:
The hospitals covered include those listed in the approved list of Medical Facilities where at least one of the hospitals is covered by your Plan. Additionally, some procedures can be carried out in hospitals that require a higher level of cover than you hold under your Plan, but these will be covered according to the level of benefit specified in your Table of Benefits.

Top chunks:

-------------------
Chunk ID: 76
Score: 0.5555
if the treatment is given in an emergency. You will need to pay any shortfall. We will pay the costs if: 1. Your Consultant obtains Our prior approval on Your behalf, and 2. You satisfy the criteria as set out in the Schedule of Benefits. 1. We will cover such costs where: a. a doctor certifies it i

-------------------
Chunk ID: 74
Score: 0.5547
ers to hospitals; and 2. transfers to ICUs in hospitals that require a higher level of cover than You hold under Your Plan. The level of Benefits payable, if any, will 

## Step 13: Backend function for Others

Others can connect a frontend to this function later.

In [31]:
def chatbot_api(user_message):
    result = answer_question(user_message)

    return {
        "answer": result["answer"],
        #"sources": result["retrieved_chunks"]
    }


reply = chatbot_api("What are the exclusions in this policy?")
print(reply["answer"])

The exclusions in this policy include:

1. Out-Patient consultations in a Consultant's room or Out-Patient setting.
2. Substance Accidents (illness or accidents arising from drinking alcohol or taking drugs).
3. Self-Harm incidents (arising from intentionally injuring oneself).
4. High Risk Activities (any high-risk activities or dangerous sports).
5. Negligence injuries caused by your own negligence.
6. Safety Equipment injuries.


## Step 14: Simple frontend (lab task)

A small interface built with `ipywidgets`:

- one input box
- one Ask button
- one answer area
- optional source display (tick **Show sources**)

In [32]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_box = widgets.Text(
    placeholder="Ask a question about the policy...",
    layout=widgets.Layout(width="80%")
)
ask_button = widgets.Button(description="Ask", button_style="primary")
show_sources = widgets.Checkbox(value=False, description="Show sources")
output_area = widgets.Output()


def on_ask_clicked(_):
    question = question_box.value.strip()

    with output_area:
        clear_output()

        if not question:
            print("Please type a question first.")
            return

        print("Thinking...")
        try:
            result = answer_question(question)
        except Exception as e:
            clear_output()
            print("Error:", e)
            return

        clear_output()
        print("Q:", result["question"])
        print("\nA:", result["answer"])

        if show_sources.value:
            print("\nSources:")
            for item in result["retrieved_chunks"]:
                print("\n--- Chunk", item["chunk_id"], "| score", item["score"])
                print(item["text"][:300])


ask_button.on_click(on_ask_clicked)
question_box.on_submit(on_ask_clicked)   # pressing Enter also works

display(widgets.VBox([
    widgets.HBox([question_box, ask_button]),
    show_sources,
    output_area
]))

## help

### Main idea

- load the document
- split it into chunks
- find the best chunks for the question
- send those chunks to the model
- return the answer

### Lab task

build a frontend with:
- one input box
- one ask button
- one answer area
- optional source display